# ModelTester — one notebook to test any model on any feature set

This notebook follows the brief's required structure:
1. **Describe the model & data**, with summary statistics (observation counts for estimation and forecast periods)
2. **RMSE chart: model vs AR1**, numbers printed on the bars
3. **Demonstrate the out-of-sample approach**
4. **Plots of the series used in the model**

…plus an extra section on **overfitting & the "lag" look** of the error charts.

**How to use it:** change `FEATURES` (Step 1) and `MODEL_NAME` (Step 2), then *Run All*.
Everything runs through the exact same engine and model classes as the official
pipeline (`src/`), so nothing here can drift from the real results — with the
default settings the numbers reproduce `reports/metrics_overall.csv` exactly.

## Step 1 — Import the dataset & choose features

The weekly table is built by the pipeline (`python run.py fetch build`): every
daily series sampled to Fridays (last value on or before each Friday), target =
log USDZAR, features known at each Friday.

**To test a different dataset/feature combination: edit `FEATURES` below.**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src import config, diagnostics as dx, plots

weekly = pd.read_parquet(config.DATA_PROCESSED / 'weekly.parquet')

# The feature set under test. Default = the official list. Try subsets, e.g.
# FEATURES = ['d4_log_gold', 'rate_spread', 'mr_gap']
FEATURES = list(config.FEATURE_COLS)
print('features under test:', FEATURES)

**Summary statistics** (brief requirement). Note the expanding window: the model
is first *estimated* on the pre-2021 sample, then re-estimated every Friday, so
by the last origin it trains on almost the whole table.

In [ ]:
dx.obs_counts(weekly)

In [ ]:
weekly[['usdzar', config.TARGET_COL, *FEATURES]].describe().round(3)

## Step 2 — Select the model

All models share one interface: `fit(train_df)` learns from data up to the
origin Friday; `predict(origin_row)` returns log USDZAR 4 weeks ahead.

| name | idea |
|---|---|
| `RandomWalk` | forecast = today (no features) |
| `RW+Drift` | today + average weekly trend (no features) |
| `AR1` | **official benchmark**: OLS of level in 4 weeks on level today |
| `OLS` | regression of the 4-week change on `FEATURES` |
| `Ridge` | same, coefficients shrunk toward 0 (CV-tuned inside the training window) |
| `Lasso` | same, weak features shrunk to exactly 0 (CV-tuned inside the training window) |

**Change `MODEL_NAME` to test another model.** It will be compared against AR1.

In [ ]:
from src.models.benchmarks import AR1Level, RandomWalk, RandomWalkDrift
from src.models.regressions import FundamentalsOLS, LassoModel, RidgeModel

MODEL_NAME = 'Lasso'   # <-- change me

def make_model():
    factories = {
        'RandomWalk': lambda: RandomWalk(),
        'RW+Drift':   lambda: RandomWalkDrift(),
        'AR1':        lambda: AR1Level(),
        'OLS':        lambda: FundamentalsOLS(features=FEATURES),
        'Ridge':      lambda: RidgeModel(features=FEATURES),
        'Lasso':      lambda: LassoModel(features=FEATURES),
    }
    return factories[MODEL_NAME]()

make_model()  # sanity check it constructs

## Step 3 — Demonstrate the out-of-sample approach

Two demonstrations, one structural and one experimental.

**(a) The engine physically truncates the data at each origin.** Below is the
actual source code of the backtest loop — the line
`train_df = df.loc[df.index <= origin]` is the entire guarantee: a model is
handed only rows up to its forecast Friday, then asked about 4 weeks later.

In [ ]:
import inspect
from src.backtest import run_backtest
print(inspect.getsource(run_backtest))

**(b) Live proof:** corrupt every row AFTER one origin with garbage and re-run —
the forecast made at that origin must not change at all. (The test suite runs
this same check for all models at several origins: `tests/test_backtest.py`.)

In [ ]:
import numpy as np

origin_pos = 600                       # any Friday in the table
one_origin = weekly.index[[origin_pos]]

clean = run_backtest(weekly, [make_model()], one_origin)

corrupted = weekly.copy()
future = corrupted.index > one_origin[0]
corrupted.loc[future, :] = np.random.default_rng(0).normal(100, 50, size=(future.sum(), weekly.shape[1]))
dirty = run_backtest(corrupted, [make_model()], one_origin)

same = clean['forecast_log'].iloc[0] == dirty['forecast_log'].iloc[0]
print(f"origin {one_origin[0].date()}: forecast unchanged after corrupting the future -> {same}")
assert same, 'LOOK-AHEAD BUG!'

## Step 4 — Run the walk-forward test

Expanding window over every Friday origin 2021–2025: refit → forecast 4 weeks →
record. We run the chosen model **and AR1** through the same loop.

*Takes ~2–4 min for Lasso/Ridge (261 origins × a small grid-search each).
Set `ORIGIN_EVERY = 2` for a quick half-resolution pass while experimenting.*

In [ ]:
from src.backtest import friday_origins

ORIGIN_EVERY = 1                        # 1 = official grid; 2 = every other Friday (faster)
origins = friday_origins(weekly)[::ORIGIN_EVERY]

results = run_backtest(weekly, [make_model(), AR1Level()], origins)
results = results.dropna(subset=['actual'])
print(f"{results['origin'].nunique()} origins, models: {list(results['model'].unique())}")

## Step 5 — Performance: RMSE vs the AR1 benchmark

Bar chart with the numbers printed on the bars (brief requirement), then the
full metric table and the year-by-year comparison.

In [ ]:
from src.evaluate import overall_table
overall = overall_table(results)
plots.fig_rmse_bar(overall);

In [ ]:
dx.summary(results, MODEL_NAME).round(4) if MODEL_NAME != 'AR1' else overall.round(4)

In [ ]:
dx.per_year_vs_benchmark(results, MODEL_NAME).round(4)

## Step 6 — Plots of the series used in the model

The target with the evaluation window shaded, then each feature in `FEATURES`
(orange band = forecast window).

In [ ]:
plots.fig_history(weekly);

In [ ]:
plots.fig_predictors(weekly, cols=FEATURES);

## Step 7 — Overfitting & the "lag" look of the error charts

When you plot |error| of the model and AR1 by origin, the two lines look almost
identical and seem to move in slow waves. Three checks below let you verify
**why**, yourself:

1. **Error correlation** — if it is ≈1.0, the models make essentially the *same*
   errors. That happens because both predict very small moves, so each error is
   ≈ (minus) the *actual* 4-week move — the wiggles you see are the rand itself,
   not model behaviour.
2. **Predicted-vs-actual scatter** — if the cloud hugs the horizontal axis, the
   model barely commits to any move: the *opposite* of overfitting in its
   forecasts. (Compare `sd(pred)/sd(actual)` across models.)
3. **Error autocorrelation** — consecutive weekly origins forecast 4-week windows
   that overlap by 3 weeks, so neighbouring errors share most of their outcome.
   Theory: autocorrelation ≈ high at lags 1–3, ≈ 0 from lag 4. That overlap is
   the whole "lag/wave" effect — and it is also exactly why our Diebold–Mariano
   test uses an autocorrelation-robust variance.
4. **In-sample vs out-of-sample RMSE** — the classic overfitting test: a ratio
   well above 1 means the model memorised noise. (Compare OLS to Lasso here.)

In [ ]:
print(f"corr({MODEL_NAME} errors, AR1 errors) = {dx.error_correlation(results, MODEL_NAME):.4f}")
dx.fig_move_scatter(results, MODEL_NAME);

In [ ]:
dx.error_autocorr(results, MODEL_NAME)

In [ ]:
# a couple of seconds: one extra fit on the pre-2021 sample
dx.insample_vs_oos(weekly, results, MODEL_NAME, make_model)

In [ ]:
dx.fig_error_timeline(results, MODEL_NAME);   # the chart from the discussion, for reference

## Step 8 — Our observations

*(write up what you find when varying `FEATURES` / `MODEL_NAME`)*

- ...
